# Question transformations

## What this notebook does

This notebook explores question-transformation techniques that improve retrieval before the final answer is generated. It starts from a Chroma vector store built on Wikivoyage content about UK destinations, then compares several ways to reformulate the user query so retrieval aligns better with the stored chunks.

The workflow moves from a baseline search with the original question to progressively stronger approaches: Rewrite-Retrieve-Read, multiple query generation, step-back prompting, and HyDE. The goal is to show that retrieval quality often depends as much on query formulation as on chunking and embeddings.


In [1]:
# Select the LLM provider for the notebook: "openai", "ollama", or "gemini".
# Values are loaded from the project-root .env file when present.
import os
import getpass
from pathlib import Path

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings


def load_project_env(filename=".env", override=True):
    current = Path.cwd().resolve()
    for directory in [current, *current.parents]:
        env_path = directory / filename
        if env_path.exists():
            for line in env_path.read_text(encoding="utf-8").splitlines():
                line = line.strip()
                if not line or line.startswith("#") or "=" not in line:
                    continue
                key, value = line.split("=", 1)
                key = key.strip()
                value = value.strip().strip('"').strip("'")
                if override or key not in os.environ:
                    os.environ[key] = value
            return env_path
    return None


PROJECT_ENV_PATH = load_project_env(override=True)

# LangSmith tracing is configured before any LangChain objects are created.
os.environ.setdefault("LANGSMITH_ENDPOINT", "https://api.smith.langchain.com")
os.environ.setdefault("LANGSMITH_PROJECT", "question-transformations")
if os.getenv("LANGSMITH_API_KEY"):
    os.environ["LANGSMITH_TRACING"] = os.getenv("LANGSMITH_TRACING", "true")

LLM_PROVIDER = os.getenv("LLM_PROVIDER", "ollama").strip().lower()

OPENAI_MODEL = os.getenv("OPENAI_MODEL", "gpt-5-nano")
OPENAI_EMBEDDING_MODEL = os.getenv("OPENAI_EMBEDDING_MODEL", "text-embedding-3-small")

OLLAMA_MODEL = os.getenv("OLLAMA_MODEL", "gemma4:e2b")
OLLAMA_EMBEDDING_MODEL = os.getenv("OLLAMA_EMBEDDING_MODEL", "nomic-embed-text")
OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434")

GEMINI_MODEL = os.getenv("GEMINI_MODEL", "gemini-flash-latest")
GEMINI_EMBEDDING_MODEL = os.getenv("GEMINI_EMBEDDING_MODEL", "gemini-embedding-2-preview")

SUPPORTED_LLM_PROVIDERS = {"openai", "ollama", "gemini"}
if LLM_PROVIDER not in SUPPORTED_LLM_PROVIDERS:
    raise ValueError(f"Unsupported LLM_PROVIDER: {LLM_PROVIDER}. Use one of: {sorted(SUPPORTED_LLM_PROVIDERS)}")


def _get_api_key(*names):
    for name in names:
        value = os.getenv(name)
        if value:
            return value
    return getpass.getpass(f"Enter your {names[0]}")


class GeminiEmbeddingsOneByOne:
    def __init__(self, embeddings):
        self.embeddings = embeddings

    def embed_documents(self, texts):
        # Gemini embedding models can return one vector per batch with this
        # LangChain version. Chroma expects one vector per document, so embed
        # each chunk separately.
        return [
            self.embeddings.embed_documents([text], batch_size=1)[0]
            for text in texts
        ]

    def embed_query(self, text):
        return self.embeddings.embed_query(text)


def ensure_ollama_model_available(model_name):
    import ollama

    try:
        installed_models = {model.model for model in ollama.list().models}
    except Exception as exc:
        raise RuntimeError(
            f"Could not connect to Ollama at {OLLAMA_BASE_URL}. "
            "Start Ollama before running this notebook."
        ) from exc

    model_aliases = {name.removesuffix(":latest") for name in installed_models}
    if model_name not in installed_models and model_name not in model_aliases:
        raise RuntimeError(
            f'Ollama model "{model_name}" is not installed. '
            f'Run `ollama pull {model_name}` in a terminal, then restart the notebook kernel.'
        )


def get_embeddings_model():
    if LLM_PROVIDER == "openai":
        return OpenAIEmbeddings(
            model=OPENAI_EMBEDDING_MODEL,
            openai_api_key=_get_api_key("OPENAI_API_KEY"),
        )
    if LLM_PROVIDER == "ollama":
        ensure_ollama_model_available(OLLAMA_EMBEDDING_MODEL)
        return OllamaEmbeddings(
            model=OLLAMA_EMBEDDING_MODEL,
            base_url=OLLAMA_BASE_URL,
        )
    if LLM_PROVIDER == "gemini":
        embeddings = GoogleGenerativeAIEmbeddings(
            model=GEMINI_EMBEDDING_MODEL,
            google_api_key=_get_api_key("GEMINI_API_KEY", "GOOGLE_API_KEY"),
        )
        return GeminiEmbeddingsOneByOne(embeddings)


def get_chat_model():
    if LLM_PROVIDER == "openai":
        return ChatOpenAI(
            model=OPENAI_MODEL,
            openai_api_key=_get_api_key("OPENAI_API_KEY"),
        )
    if LLM_PROVIDER == "ollama":
        ensure_ollama_model_available(OLLAMA_MODEL)
        return ChatOllama(
            model=OLLAMA_MODEL,
            base_url=OLLAMA_BASE_URL,
        )
    if LLM_PROVIDER == "gemini":
        return ChatGoogleGenerativeAI(
            model=GEMINI_MODEL,
            google_api_key=_get_api_key("GEMINI_API_KEY", "GOOGLE_API_KEY"),
        )


def message_text(message):
    content = getattr(message, "content", message)
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, dict) and "text" in item:
                parts.append(str(item["text"]))
            else:
                parts.append(str(item))
        return "\n".join(parts)
    return str(content)


print(f"Loaded env from: {PROJECT_ENV_PATH or 'not found'}")
print(f"Using {LLM_PROVIDER} chat model and matching embeddings")
if LLM_PROVIDER == "ollama":
    print(f"Ollama chat model: {OLLAMA_MODEL}")
elif LLM_PROVIDER == "gemini":
    print(f"Gemini chat model: {GEMINI_MODEL}")
print(f"LangSmith tracing: {os.getenv('LANGSMITH_TRACING', 'false')} | project: {os.getenv('LANGSMITH_PROJECT')}")


Loaded env from: /home/thimoty/git/building-llm-applications/.env
Using gemini chat model and matching embeddings
Gemini chat model: gemini-flash-latest
LangSmith tracing: true | project: qachatbot


# Splitting and ingesting the content of various URLs (across UK destinations)

### Why the notebook rebuilds the UK collections

The retrieval experiments in this notebook rely on a shared dataset of Wikivoyage pages about UK destinations. Rebuilding the Chroma collections at the start keeps the later comparisons fair: each technique works against the same chunks, the same embeddings, and the same retrieval surface.

That matters because the goal here is query transformation, not changing the corpus. The retrieval improvements shown later come from reformulating the question, while the indexed content stays constant.


### Preparing the Chroma DB collections

In [2]:
from langchain_chroma import Chroma


In [3]:
uk_granular_collection = Chroma(
    collection_name="uk_granular",
    embedding_function=get_embeddings_model(),
)

uk_granular_collection.reset_collection() #A

### Splitting and ingesting HTML content with the HTMLSectionSplitter 

In [4]:
from langchain_text_splitters import HTMLSectionSplitter
from langchain_community.document_loaders import AsyncHtmlLoader

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [5]:
uk_destinations = [
    "Cornwall", "North_Cornwall", "South_Cornwall", "West_Cornwall", 
    "Tintagel", "Bodmin", "Wadebridge", "Penzance", "Newquay",
    "St_Ives", "Port_Isaac", "Looe", "Polperro", "Porthleven",
    "East_Sussex", "Brighton", "Battle", "Hastings_(England)", 
    "Rye_(England)", "Seaford", "Ashdown_Forest"
]

wikivoyage_root_url = "https://en.wikivoyage.org/wiki"

In [6]:
uk_destination_urls = [f'{wikivoyage_root_url}/{d}' 
                       for d in uk_destinations]

In [7]:
headers_to_split_on = [("h1", "Header 1"),("h2", "Header 2")]
html_section_splitter = HTMLSectionSplitter(
    headers_to_split_on=headers_to_split_on)

In [8]:
def split_docs_into_granular_chunks(docs):
    all_chunks = []
    for doc in docs:
        html_string = doc.page_content #B
        temp_chunks = html_section_splitter.split_text(
            html_string) #C
        h2_temp_chunks = [chunk for chunk in 
                          temp_chunks if "Header 2" 
                          in chunk.metadata] #D
        all_chunks.extend(h2_temp_chunks) 

    return all_chunks

In [9]:
for destination_url in uk_destination_urls:
    html_loader = AsyncHtmlLoader(
        destination_url) #E
    docs =  html_loader.load() #F
    
    for doc in docs:
        print(doc.metadata)
        granular_chunks = split_docs_into_granular_chunks(docs)
        uk_granular_collection.add_documents(
            documents=granular_chunks)

#A In case it exists
#B Extract the HTML text from the document
#C Each chunk is a H1 or H2 HTML section
#D Only keep content associated with H2 sections        
#E Loader for one destination
#F Documents of one destination

Fetching pages: 100%|##########| 1/1 [00:00<00:00,  7.29it/s]


{'source': 'https://en.wikivoyage.org/wiki/Cornwall', 'title': 'Cornwall – Travel guide at Wikivoyage', 'language': 'en'}


Fetching pages: 100%|##########| 1/1 [00:00<00:00, 11.29it/s]


{'source': 'https://en.wikivoyage.org/wiki/North_Cornwall', 'title': 'North Cornwall – Travel guide at Wikivoyage', 'language': 'en'}


Fetching pages: 100%|##########| 1/1 [00:00<00:00, 13.24it/s]


...

{'source': 'https://en.wikivoyage.org/wiki/Rye_(England)', 'title': 'Rye (England) – Travel guide at Wikivoyage', 'language': 'en'}


Fetching pages: 100%|##########| 1/1 [00:00<00:00, 13.18it/s]


{'source': 'https://en.wikivoyage.org/wiki/Seaford', 'title': 'Seaford – Travel guide at Wikivoyage', 'language': 'en'}


Fetching pages: 100%|##########| 1/1 [00:00<00:00, 10.56it/s]


{'source': 'https://en.wikivoyage.org/wiki/Ashdown_Forest', 'title': 'Ashdown Forest – Travel guide at Wikivoyage', 'language': 'en'}


# Rewrite-retrieve-read

### Why this notebook starts from retrieval quality

This notebook focuses on a practical problem in RAG systems: a vector store may be well populated, but retrieval still underperforms when the user question is vague, broad, or phrased differently from the indexed documents. The point of these techniques is not to change the knowledge base, but to transform the question into a better retrieval input.

The baseline search in this section is intentionally simple. It gives you a reference point before introducing query-transformation methods that improve semantic alignment between the question and the stored chunks.


## Retrieving content with original user question

In [10]:
user_question = "Tell me some fun things I can enjoy in Cornwall"
initial_results = uk_granular_collection.similarity_search(
    query=user_question,k=4)
for doc in initial_results:
    print(doc)

page_content='Do 
 [ edit ] 
 
 Cornwall, in particular Newquay, is the UK's  surfing  capital, with equipment hire and surf schools present on many of the county's beaches, and events like the UK championships or Boardmasters festival. 
 The  South West Coast Path  runs along the coastline of Britain's south-west peninsula. The Cornish section is supposed to be the most scenic (unless you talk to someone in Devon, in which case the Devon part is most scenic). It is particularly scenic around Penwith and the Lizard. The trail takes walkers to busy towns, remote cliffs, beaches, heaths, farms and fishing villages. Walking along it is a great way to experience the region in all its variety. (Walking the entire path takes several weeks, walking on a choice part of it is easier.) 
 The  Camel Trail  is an  18-mile (29   km)  off-road cycle-track that follows the route of a former railway line along the scenic estuary of the river Camel from  Padstow  to Wenford Bridge via  Wadebridge  and 

In [11]:
# COMMENT: the retrieval from the vector store against the original question is bad

## Question rewrite

### Why rewrite the question before retrieval

Rewrite-Retrieve-Read uses an LLM to convert the original user request into a cleaner search query for the vector store, while keeping the original wording for answer synthesis. This separation matters: the retriever benefits from specificity and explicit search terms, whereas the final answer should still respond to the user as they asked the question.

In practice, query rewriting helps when the initial request is underspecified or uses wording that does not map cleanly to the embeddings space. The rewritten query is optimized for retrieval, not for user-facing dialogue.


### Setting up the query rewriter chain

In [12]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate

In [13]:
llm = get_chat_model()

In [14]:
rewriter_prompt_template = """
Generate search query for the Chroma DB vector store
from a user question, allowing for a more accurate 
response through semantic search.
Just return the revised Chroma DB query, with quotes around it. 

User question: {user_question}
Revised Chroma DB query:
"""

rewriter_prompt = ChatPromptTemplate.from_template(
    rewriter_prompt_template) 

In [15]:
rewriter_chain = rewriter_prompt | llm | StrOutputParser()

### Retrieving content with the rewritten query

In [16]:
user_question ="Tell me some fun things I can do in Cornwall"

search_query = rewriter_chain.invoke(
    {"user_question": user_question})
print(search_query)

"popular tourist attractions and fun activities to do in Cornwall"


In [17]:
improved_results = uk_granular_collection.similarity_search(
    query=search_query,k=3)
for doc in improved_results:
    print(doc)

page_content='Do 
 [ edit ] 
 
 Cornwall, in particular Newquay, is the UK's  surfing  capital, with equipment hire and surf schools present on many of the county's beaches, and events like the UK championships or Boardmasters festival. 
 The  South West Coast Path  runs along the coastline of Britain's south-west peninsula. The Cornish section is supposed to be the most scenic (unless you talk to someone in Devon, in which case the Devon part is most scenic). It is particularly scenic around Penwith and the Lizard. The trail takes walkers to busy towns, remote cliffs, beaches, heaths, farms and fishing villages. Walking along it is a great way to experience the region in all its variety. (Walking the entire path takes several weeks, walking on a choice part of it is easier.) 
 The  Camel Trail  is an  18-mile (29   km)  off-road cycle-track that follows the route of a former railway line along the scenic estuary of the river Camel from  Padstow  to Wenford Bridge via  Wadebridge  and 

### Combining everything in a single RAG chain

In [18]:
from langchain_core.runnables import RunnablePassthrough

In [19]:
retriever = uk_granular_collection.as_retriever()

rag_prompt_template = """
Given a question and some context, answer the question.
If you do not know the answer, just say I do not know.

Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(
    rag_prompt_template) 

rewrite_retrieve_read_rag_chain = (
    {
        "context": {"user_question": RunnablePassthrough()} 
            | rewriter_chain | retriever,#A
        "question": RunnablePassthrough(),#B
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)
#A The context is returned by the retriver after feeding to it the rewritten query
#B This is the original user question

In [20]:
user_question = "Tell me some fun things I can do in Cornwall"

answer = rewrite_retrieve_read_rag_chain.invoke(user_question)
print(answer)

Based on the context provided, here are several fun things you can do in Cornwall:

**Outdoor Activities and Adventure**
*   **Surfing:** Visit Newquay, the UK's surfing capital, to take lessons or attend events like the Boardmasters festival.
*   **Walking:** Explore the South West Coast Path for scenic views of cliffs, beaches, and fishing villages, particularly around Penwith and the Lizard.
*   **Cycling:** Ride the Camel Trail, an 18-mile off-road cycle track between Padstow and Wenford Bridge.
*   **Theme Parks:** Visit Camel Creek Adventure Park in Wadebridge for a family day out.
*   **Zip Lining:** Experience the massive zip line at the Eden Project near St Austell.

**Sightseeing and Attractions**
*   **Gardens and Nature:** Visit the Eden Project to see a massive collection of flora in space-age domes, or explore the 80-acre Lost Gardens of Heligan.
*   **Historic Sites:** Visit Tintagel, the legendary birthplace of King Arthur, or explore National Trust properties like the 

# Multiple query generation with MultiQueryRetriever

### Why generate multiple queries

A single rewritten query is useful when the original question is weakly phrased, but it can still miss relevant context when the request contains multiple implicit angles. Multi-query retrieval addresses this by generating several alternative formulations of the same intent, retrieving documents for each variant, and then merging the resulting context.

This is useful when different chunks may be reachable through different phrasings. In other words, instead of betting on one perfect query, the retriever explores several semantic perspectives on the same question.


In [21]:
from langchain_classic.retrievers.multi_query import MultiQueryRetriever
from langchain_core.prompts import ChatPromptTemplate

from typing import List
from langchain_core.output_parsers import BaseOutputParser
from pydantic import BaseModel, Field

## Implementing a custom MultiQueryRetriver

### Setting up the prompt

In [22]:
multi_query_gen_prompt_template = """
You are an AI language model assistant. Your task 
is to generate five different versions of the given 
user question to retrieve relevant documents from a vector 
database. By generating multiple perspectives on the user 
question, your goal is to help the user overcome some of 
the limitations of the distance-based similarity search. 
Provide these alternative questions separated by newlines.
Original question: {question}
"""

multi_query_gen_prompt = ChatPromptTemplate.from_template(
    multi_query_gen_prompt_template) 

### Setting up the multi-query parser

In [23]:
class LineListOutputParser(BaseOutputParser[List[str]]):
    """Parse out a question from each output line."""

    def parse(self, text: str) -> List[str]:
        lines = text.strip().split("\n")
        return list(filter(None, lines))  

questions_parser = LineListOutputParser()

### Setting up the chain to generate multiple queries

In [24]:
llm = get_chat_model()

In [25]:
multi_query_gen_chain = multi_query_gen_prompt | llm | questions_parser

### Testing the Multi query gen chain

In [26]:
user_question = "Tell me some fun things I can do in Cornwall"

multiple_queries = multi_query_gen_chain.invoke(user_question)

In [27]:
multiple_queries

['What are the most popular tourist attractions and leisure activities for visitors in Cornwall?',
 'Can you recommend some unique outdoor adventures and scenic landmarks to explore in Cornwall?',
 'What are the best family-friendly things to do and places to visit during a trip to Cornwall?',
 'Could you provide a list of top-rated cultural experiences and historic sites in the Cornwall area?',
 'What are some hidden gems and off-the-beaten-path activities for entertainment in Cornwall?']

### Setting up the MultiQueryRetriever

In [28]:
basic_retriever = uk_granular_collection.as_retriever()

multi_query_retriever = MultiQueryRetriever(
    retriever=basic_retriever, llm_chain=multi_query_gen_chain, 
    parser_key="lines" #A
)  
#A this is the key for the parsed output

### Using the multi_query retriever

In [29]:
user_question = "Tell me some fun things I can do in Cornwall"

retrieved_docs = multi_query_retriever.invoke(user_question)

In [30]:
retrieved_docs

[Document(id='73e0a6a9-d435-4d29-8e3b-9712b43a1aa9', metadata={'Header 2': 'See'}, page_content="See \n [ edit ] \n \n The Cheesewring at Minions,  Bodmin Moor . \n St. Michael's Mount lies offshore close to Penzance. \n Cornwall boasts many attractions for the traveller, many lying outside of cities and towns amidst the Cornish landscape: \n \n Within the 208 m² of the  Bodmin Moor , is  King Arthur's Hall , a megalithic monument and  Brown Willy , the highest point in Cornwall at 417 m (1,368 ft).  Dozmary Pool  is a small beautiful lake where, according to legend, King Arthur was entrusted with the sword Excalibur. There is also a reputed  Beast of the Moor , a large wild-cat that haunts and stalks at night, but is similar in fantasy to the Loch Ness Monster, in that no one can prove it exists, though sightings, theories and track-marks abound. \n The  Eden Project , near  St Austell , a fabulous collection of flora from all over the planet housed in two 'space age' transparent dome

## Using directly a standard MultiQueryRetriever 

In [31]:
std_multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=basic_retriever, llm=llm
)

In [32]:
user_question = "Tell me some fun things I can do in Cornwall"

retrieved_docs = std_multi_query_retriever.invoke(user_question)

In [33]:
retrieved_docs

[Document(id='712158cf-0bc7-43bf-bafd-0517056d0466', metadata={'Header 2': 'Do'}, page_content='Do \n [ edit ] \n \n Cornwall, in particular Newquay, is the UK\'s  surfing  capital, with equipment hire and surf schools present on many of the county\'s beaches, and events like the UK championships or Boardmasters festival. \n The  South West Coast Path  runs along the coastline of Britain\'s south-west peninsula. The Cornish section is supposed to be the most scenic (unless you talk to someone in Devon, in which case the Devon part is most scenic). It is particularly scenic around Penwith and the Lizard. The trail takes walkers to busy towns, remote cliffs, beaches, heaths, farms and fishing villages. Walking along it is a great way to experience the region in all its variety. (Walking the entire path takes several weeks, walking on a choice part of it is easier.) \n The  Camel Trail  is an  18-mile (29 \xa0 km)  off-road cycle-track that follows the route of a former railway line along

# Step-back question

### Setting up the chain to generate the step-back question

### Why step-back prompting helps

Step-back prompting broadens the user question before retrieval. Instead of searching only for the immediate request, the system first asks a more general question that captures the surrounding conceptual context, then uses that broader context to support the final answer.

This approach is useful when the original question is too narrow to retrieve the foundations needed for a good response. The broader query often surfaces documents that explain the domain, constraints, or high-level recommendations behind the final answer.


In [34]:
llm = get_chat_model()

In [35]:
step_back_prompt_template = """
Generate a less specific question (aka Step-back question) 
for the following detailed question, so that a wider context 
can be retrieved.
Detailed question: {detailed_question}
Step-back question:
"""

step_back_prompt = ChatPromptTemplate.from_template(
    step_back_prompt_template) 

In [36]:
step_back_question_gen_chain = step_back_prompt | llm | StrOutputParser()

### Testing the step-back-question generation chain

In [37]:
user_question = "Can you give me some tips for a trip to Brighton?"

step_back_question = step_back_question_gen_chain.invoke(user_question)

In [38]:
step_back_question

'Step-back question: **What are the general travel considerations and highlights for visiting coastal cities in the South of England?**'

### Incorporating step-back question generation chain into the RAG chain

In [39]:
retriever = uk_granular_collection.as_retriever()

rag_prompt_template = """
Given a question and some context, answer the question.
If you do not know the answer, just say I do not know.

Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(rag_prompt_template) 

step_back_question_rag_chain = (
    {
        "context": {"detailed_question": RunnablePassthrough()} 
           | step_back_question_gen_chain | retriever,#A
        "question": RunnablePassthrough(),#B
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)
#A The context is returned by the retriver after feeding to it the step-back question
#B This is the original user question

In [40]:
user_question = "Can you give me some tips for a trip to Brighton?"

answer = step_back_question_rag_chain.invoke(user_question)
print(answer)

Based on the provided text, here are some tips for a trip to Brighton:

*   **Best Time to Visit:** While it is a year-round destination, the city comes to life in the **springtime**. Specifically, **May** is a great time to visit for the Brighton Festival and the Festival Fringe. **Summer** is ideal for enjoying the five-mile stretch of shingle beach and sunsets.
*   **What to See:** Look for the grand **Regency architecture** and landmarks with oriental-inspired styles, such as the **Grade-I Listed Pavilion** (designed by John Nash). There are also various art galleries to explore.
*   **Vibe and Culture:** Brighton is known as the "gay capital of Britain" and has a significant gay district in **Kemp Town**, which contributes to the city's Bohemian atmosphere. It is often referred to as "London-by-the-Sea" because of its popularity with media and music professionals.
*   **Activities:** You can spend time on the **shingle beach**, which faces south onto the English Channel. The city 

# Hypotetical DocumentEmbeddings (HyDE)

### Setting up the chain to generate the hypotetical document associated to the user question

In [41]:
llm = get_chat_model()

### Why HyDE can improve retrieval

HyDE, or Hypothetical Document Embeddings, generates a plausible answer-like passage from the user question and uses that generated text as the retrieval query. The intuition is that a synthetic answer may resemble the wording and structure of the indexed chunks better than the raw user question does.

This technique is particularly useful when the user asks in abstract language but the source material is concrete and descriptive. By retrieving against a generated pseudo-document, the vector store can match passages that are semantically closer to the kind of answer the system is trying to assemble.


In [42]:
hyde_prompt_template = """
Write one sentence that could answer the provided question. 
Do not add anything else.
Question: {question}
Sentence:
"""

hyde_prompt = ChatPromptTemplate.from_template(hyde_prompt_template)

In [43]:
hyde_chain = hyde_prompt | llm | StrOutputParser()

### Testing the hyde generation chain

In [44]:
user_question = "What are the best beaches in Cornwall?"

hypotetical_document = hyde_chain.invoke(user_question)

In [45]:
hypotetical_document

'Some of the most highly-regarded beaches in Cornwall include Kynance Cove, Porthcurno, Fistral Beach, and Porthmeor Beach.'

### Incorporating hyde chain into the RAG chain

In [46]:
retriever = uk_granular_collection.as_retriever()

rag_prompt_template = """
Given a question and some context, answer the question.
Only use the provided context to answer the question.
If you do not know the answer, just say I do not know. 

Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(rag_prompt_template) 

hyde_rag_chain = (
    {
        "context": {"question": RunnablePassthrough()} 
           | hyde_chain | retriever,#A
        "question": RunnablePassthrough(),#B
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)
#A The context is returned by the retriver after feeding to it the hypotetical document
#B This is the original user question

In [47]:
user_question = "What are the best beaches in Cornwall?"

answer = hyde_rag_chain.invoke(user_question)
print(answer)

According to the provided context, beaches that form an important part of Cornwall's tourist industry include:

*   **North Coast:** Bude, Polzeath, Watergate Bay, Perranporth, Porthtowan, Fistral Beach, Newquay, St Agnes, and St Ives.
*   **South Coast:** Gyllyngvase beach in Falmouth and Praa Sands.

The context also notes that Falmouth is "famous for its beaches."
